In [ ]:
# 必要なライブラリのインストール
!pip install matplotlib-venn torchinfo transformers

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from matplotlib_venn import venn2
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import os
from transformers import ViTForImageClassification

# --- 1. Google Driveの設定 ---
print("Mounting Google Drive...")

# 保存先フォルダ（MyDrive直下の 'Research_Experiment' フォルダに保存します）
# 必要に応じて変更してください
SAVE_DIR = os.path.abspath("artifacts/research_experiment")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

# 保存ファイルパス
RESULT_JSON_PATH = os.path.join(SAVE_DIR, "cifar100_difficulty_labels.json")
GRAPH_IMAGE_PATH = os.path.join(SAVE_DIR, "difficulty_venn_diagram.png")

print(f"Results will be saved to: {SAVE_DIR}")

# --- 2. 設定 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

FORCE_INFERENCE = False  # Trueなら既存ファイルがあっても強制的に再推論

# --- 3. データセット準備 ---
print("Preparing Dataset...")

# Lowモデル用 (32x32, CIFAR標準正規化)
transform_low = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
])

# Highモデル用 (224x224, ImageNet標準正規化)
transform_high = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 生データ (画像ID管理用)
testset_raw = torchvision.datasets.CIFAR100(root=os.environ.get('CODEX_COLAB_DATA_DIR', './data'), train=False, download=True, transform=None)

# 修正: PIL画像をリストのまま受け取るための設定
def custom_collate(batch):
    images = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])
    return images, labels

testloader = DataLoader(testset_raw, batch_size=50, shuffle=False, collate_fn=custom_collate)

# --- 4. 推論実行関数 ---
def run_inference():
    print("Loading Models...")

    # Low: MobileNetV2 x0.5
    try:
        low_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_mobilenetv2_x0_5", pretrained=True)
    except:
        print("GitHub load failed. Using local fallback.")
        return []
    low_model.to(device).eval()

    # High: ViT-Base
    try:
        high_model = ViTForImageClassification.from_pretrained("Ahmed9275/Vit-Cifar100")
    except:
        print("HuggingFace load failed.")
        return []
    high_model.to(device).eval()

    results = []
    print("Starting Inference on CIFAR-100 Testset (10,000 images)...")

    # バッチ処理で推論
    global_idx = 0
    with torch.no_grad():
        for batch in tqdm(testloader):
            images, labels = batch # ここでのimagesはPIL形式のリスト
            labels = labels.to(device)

            # --- Lowモデル推論 ---
            # PIL -> Tensor変換
            batch_low = torch.stack([transform_low(img) for img in images]).to(device)
            out_low = low_model(batch_low)
            _, pred_low = torch.max(out_low, 1)

            # --- Highモデル推論 ---
            batch_high = torch.stack([transform_high(img) for img in images]).to(device)
            out_high = high_model(batch_high).logits
            _, pred_high = torch.max(out_high, 1)

            # 結果格納
            for i in range(len(labels)):
                label_val = labels[i].item()
                low_val = pred_low[i].item()
                high_val = pred_high[i].item()

                is_low_correct = (low_val == label_val)
                is_high_correct = (high_val == label_val)

                # 難易度カテゴリの決定
                if is_low_correct and is_high_correct:
                    cat = "Easy"
                elif not is_low_correct and is_high_correct:
                    cat = "Hard"
                elif not is_low_correct and not is_high_correct:
                    cat = "Impossible"
                else:
                    cat = "Inverse" # Lowのみ正解

                results.append({
                    "index": global_idx,
                    "label": label_val,
                    "low_pred": low_val,
                    "high_pred": high_val,
                    "low_correct": is_low_correct,
                    "high_correct": is_high_correct,
                    "category": cat
                })
                global_idx += 1

    return results

# --- 5. メイン処理 ---

if os.path.exists(RESULT_JSON_PATH) and not FORCE_INFERENCE:
    print(f"Loading existing results from '{RESULT_JSON_PATH}'...")
    with open(RESULT_JSON_PATH, 'r') as f:
        inference_data = json.load(f)
else:
    inference_data = run_inference()
    if inference_data:
        print(f"Saving results to '{RESULT_JSON_PATH}'...")
        with open(RESULT_JSON_PATH, 'w') as f:
            json.dump(inference_data, f, indent=2)
